# Решение задачи классификации с помощью kNN

## Цель и задачи  

В этом практическом задании вам предлагается самостоятельно обучить модель kNN для работы с многоклассовой классификацией. 
 
Вы уже знакомы с данными - это тот же датасет `song_data.csv`, содержащий характеристики музыкальных треков с платформы Spotify. Однако на этот раз нужно научиться предсказывать не рейтинг популярности, а дискретную целевую переменную (какую именно, вы можете выбрать сами).


**Цель:** Решить задачу классификации, провести эксперименты с параметрами, сравнить с базовой моделью и линейными моделями.   

При выполнении задания вы можете придерживаться предложенного плана или определить шаги самостоятельно.

**Рекомендуемый план работы:**

1. Загрузить данные и выбрать целевую переменную.
2. Провести предобработку данных (масштабирование).
3. Построить базовую модель.
4. Подобрать оптимальные гиперпараметры с помощью GridSearch:
    - n_neighbors
    - metric
    - weights
5. Обучить модель с лучшими гиперпараметрами и сравнить её с базовой.
6. Обучить модель логистической регресии и сравнить результаты с kNN.

## Данные

Давайте вспомним, какие признаки есть в датасете `song_data.csv`.

### Описание данных

- `song_duration_ms` — длительность трека в миллисекундах.
- `acousticness` — оценка акустичности (показывает, как много в треке "живой", а не электронной музыки). 
- `danceability` — оценка "танцевальности" трека. 
- `energy` — оценка энергичности трека. 
- `instrumentalness` — вероятность того, что трек не содержит вокал.
- `key` — музыкальный ключ, в котором написан трек (число от 0 до 11, соответствующее музыкальным нотам).
- `liveness` — вероятность того, что трек записан “вживую”. 
- `loudness` — громкость трека в децибелах.
- `audio_mode` — лад трека: 1 — мажорный, 0 — минорный. 
- `speechiness` — оценка “разговорности”: чем выше значение, тем больше в треке речевых фрагментов вместо пения.
- `tempo` — темп музыки в ударах в минуту (BPM).
- `audio_valence` — оценка "позитивности" настроения трека. 
- `time_signature` — ритмичность трека.
- `song_popularity` — индекс популярности трека на Spotify.

----

## 1. Загрузка данных и выделение целевой переменной

Прежде, чем переходить к работе, откроем датасет `song_data.csv` и очистим его от выбросов (логику обработки аномальных значений мы обсуждали в предыдущем уроке).

In [36]:
import pandas as pd
import numpy as np
import phik

# Библиотеки машинного обучения
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, TargetEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.frozen import FrozenEstimator
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, \
    average_precision_score, log_loss, recall_score, precision_score, brier_score_loss, \
    confusion_matrix
from sklearn.model_selection import cross_validate, StratifiedKFold, GridSearchCV
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, RFE
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier

# Библиотеки для построения графиков
import seaborn as sns
import matplotlib.pyplot as plt

# Сериализация пайплайна
import joblib

# Для форматирования в HTML
from IPython.display import display, HTML, Markdown

In [16]:
# Загружаем данные
df = pd.read_csv('datasets/song_dataset.csv')

# Удаляем строки, содержащие аномальные значения
df = df[(df['song_duration_ms'] >= 60000) & (df['song_duration_ms'] <= 600000)]
df = df[(df['loudness'] >= -20) & (df['loudness'] <= 0)]
df = df[df['tempo'] >= 30]
df = df[df['time_signature'] != 0]

df.head()

,song_popularity,song_duration_ms,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,audio_mode,speechiness,tempo,time_signature,audio_valence
0,73,262333,0.005520,0.496,0.682,0.000029,8,0.0589,-4.095,1,0.0294,167.060,4,0.474
1,66,216933,0.010300,0.542,0.853,0.000000,3,0.1080,-6.407,0,0.0498,105.256,4,0.370
2,76,231733,0.008170,0.737,0.463,0.447000,0,0.2550,-7.828,1,0.0792,123.881,4,0.324
3,74,216933,0.026400,0.451,0.970,0.003550,0,0.1020,-4.938,1,0.1070,122.444,4,0.198
4,56,223826,0.000954,0.447,0.766,0.000000,10,0.1130,-5.065,1,0.0313,172.011,4,0.574


### Выбор целевой переменной

Чтобы решать задачу многоклассовой классификации, необходимо выделить признак, который может принимать одно из нескольких возможных значений (принадлежать одному из классов). 

Здесь есть несколько вариантов действий:  

1. Использовать `song_popularity`. Как и модель регрессии, модель классификации будет предсказывать популярность трека. Однако на этот раз предсказанием будет не точное число в диапазоне от 0 до 100, а "уровень" популярности (например, число от 0 до 9).  
    
    
2. Использовать другие непрерывные признаки - например, акустичность (`acousticness`), энергичность (`energy`) и т.д. Здесь вы можете поэкспериментировать - например, построить модель, которая оценивает, подходит ли трек для танцев.  
    
    
3. Использовать один готовых дискретных признаков (`key` или `time_signature`).  
  
  
Вы можете выбрать любой способ и сделать целевой переменной понравившийся признак - главное, чтобы его значения можно преобразовать в несколько категорий. Мы рекомендуем взять от 3 до 10 классов.

#### Дискретизация

Если вы хотите использовать первый или второй подход, нужно преобразовать выбранную непрерывную переменную в категориальную с ограниченным числом классов. Этот процесс называется **дискретизацией** (или **биннингом**). В `pandas` для этого удобно использовать метод `cut`.

In [17]:
# Преобразуем значения song_popularity в 10 интервалов (классов) с метками 0...9.

df['popularity_class'] = pd.cut(
    df['song_popularity'], # Непрерывный признак для преобразования
    bins=10,               # Количество классов (рекомендуется выбрать число от 3 до 10)
    labels=False,          # Eсли False, в качестве меток используются номера интервалов (0, 1, ...)
)

# Результат: непрерывная и дискретная версия признака:
print(df[['song_popularity', 'popularity_class']].head())

   song_popularity  popularity_class
0               73                 7
1               66                 6
2               76                 7
3               74                 7
4               56                 5


Далее приведено несколько примеров того, как можно подготовить целевую переменную. Вы можете выбрать понравившийся вариант или релаизовать свой по аналогии.

#### Пример 1
Преобразование непрерывного значения `song_popularity` в 3 класса.

In [18]:
df = pd.read_csv('datasets/song_dataset.csv')

df = df[(df['song_duration_ms'] >= 60000) & (df['song_duration_ms'] <= 600000)]
df = df[(df['loudness'] >= -20) & (df['loudness'] <= 0)]
df = df[df['tempo'] >= 30]
df = df[df['time_signature'] != 0]


df['song_popularity'] = pd.cut(
    df['song_popularity'],
    bins=3,               
    labels=False,          
)

# Проверим распределение классов
df['song_popularity'].value_counts()

song_popularity
1    962
2    724
0    200
Name: count, dtype: int64

#### Пример 2
Преобразование непрерывного значения `danceability` в бинарный признак.
Такой таргет будет показывать, подходит ли трек для танцев. Например, если значение признака больше 0.7, композицию можно считать "танцевальной".

In [23]:
df = pd.read_csv('datasets/song_dataset.csv')

df = df[(df['song_duration_ms'] >= 60000) & (df['song_duration_ms'] <= 600000)]
df = df[(df['loudness'] >= -20) & (df['loudness'] <= 0)]
df = df[df['tempo'] >= 30]
df = df[df['time_signature'] != 0]

# Сравниванием значение с порогом
df['is_danceable'] = (df['danceability'] > 0.7).astype(int) 
# Удаляем исходный признак
df = df.drop(columns=['danceability']) 

# Проверим распределение классов
df['is_danceable'].value_counts()

is_danceable
0    1403
1     483
Name: count, dtype: int64

### Преобразование признака

Прежде, чем двигаться дальше, выполниите дискретизацию выбранного признака.   

**Не забудьте удалить из датасета исходный признак, чтобы избежать утечки данных!**

In [25]:
# Напишите ваш код
df['song_popularity'] = pd.cut(
    df['song_popularity'],
    bins=3,               
    labels=False,          
)

X = df.drop(columns=['song_popularity'])
y = df['song_popularity']

----

## 2. Масштабирование признаков

Чтобы модель kNN работала корректно, используйте `ColumnTransformer` для масштабирования числовых признаков.  
  
**Возможные эксперименты**   
При желании вы можете провести небольшие исследования. Например:
- Обучить модель на исходных и отмасштабированных данных и сравнить качество предсказаний.
- Использовать `StandardScaler` и `MinMaxScaler` и оценить, какой подход даёт лучшие результаты.

In [26]:
numeric_features = X.columns
preprocess = ColumnTransformer([
    ('num', StandardScaler(), numeric_features)
])

----

## 3. Построение базовой модели

Базовая модель (baseline) используется для того, чтобы оценить сложность задачи и то, какие приросты дают более сложные подходы. Для этого нужно обучить класс `KNeighborsClassifier` с параметрами по умолчанию и посчитать качество (не забудьте поделить данные на обучающие, валидационные и тестовые).
  
Для удобства вы можете объединить трансформер и модель в один пайплайн с помощью класса `Pipeline`.

In [44]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, stratify=y_train_val)

pipe = Pipeline([
    ('pp', preprocess),
    ('classify', KNeighborsClassifier())
])

base_pipe = clone(pipe)

base_pipe.fit(X_train_val, y_train_val)
print(classification_report(y_train, base_pipe.predict(X_train)))

              precision    recall  f1-score   support

           0       0.45      0.33      0.38       120
           1       0.66      0.80      0.72       577
           2       0.67      0.53      0.59       434

    accuracy                           0.65      1131
   macro avg       0.60      0.55      0.57      1131
weighted avg       0.64      0.65      0.64      1131



----

## 4. Подбор гиперпараметров

Подбор гиперпараметров часто позволяет значительно улучшить качетство модели. Для перебора различных комбинаций с использованием кросс-валидации используйте класс `GridSearchCV`.

Напомним, что при решении задачи регрессии вы перебирали такие наборы гиперпараметров:
- `n_neighbors` (количество соседей): [3, 5, 10, 15, 20, 25, 30].
- `weights` (способ взвешивания): [`uniform`, `distance`].
- `metric` (метрика расстояния): `minkowski` с подбором `p` = [1, 2, 3, 4]. 

Вы можете вновь использовать эти значения или выбрать свои диапазоны.

In [50]:
param_grid = {
    'classify__n_neighbors': [3, 5, 10, 15, 20, 25, 30],
    'classify__weights': ['uniform', 'distance'],
    'classify__metric': ['minkowski'],
    'classify__p': [1, 2, 3, 4]
}

grid = GridSearchCV(estimator=pipe,
                    param_grid=param_grid,
                    cv=5,
                    scoring='f1_macro')

grid.fit(X_train_val, y_train_val)
print(grid.best_params_)

{'classify__metric': 'minkowski', 'classify__n_neighbors': 15, 'classify__p': 4, 'classify__weights': 'distance'}


----

## 5. Обучение лучшей модели

После определения оптимальных гиперпараметров можно обучить лучшую версию модели и сравнить её качество с базовой версией.

In [51]:
best_pipe = clone(pipe)
best_pipe.set_params(**grid.best_params_)
best_pipe.fit(X_train_val, y_train_val)

print(classification_report(y_train_val, best_pipe.predict(X_train_val)))
print(classification_report(y_test, best_pipe.predict(X_test)))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       160
           1       1.00      1.00      1.00       769
           2       1.00      1.00      1.00       579

    accuracy                           1.00      1508
   macro avg       1.00      1.00      1.00      1508
weighted avg       1.00      1.00      1.00      1508

              precision    recall  f1-score   support

           0       0.50      0.03      0.05        40
           1       0.57      0.67      0.61       193
           2       0.49      0.50      0.49       145

    accuracy                           0.54       378
   macro avg       0.52      0.40      0.39       378
weighted avg       0.53      0.54      0.51       378



## 6. Сравнение с линейными моделями

На этом этапе вы можете оценить эффективность kNN по сравнению с другими моделями для многоклассовой классификации.

**Возможные эксперименты:** 
- Обучить одну из линейных моделей (OvR, OvO, многоклассовый логрег) с подбором гиперпараметров и сравнить качество с kNN.
- Оценить время обучения kNN и линейных моделей, а также сравнить время, необходимое для предсказания.

In [52]:
pipe_2 = Pipeline([
    ('pp', preprocess),
    ('classify', LogisticRegression())
])

lr_param_grid = [
    {
        "classify__solver": ['lbfgs'],
        "classify__C": [0.01, 0.1, 1, 10, 100],
        "classify__class_weight": [None, 'balanced']
    },
    {
        "classify__solver": ['saga'],
        "classify__penalty": ["l1"],
        "classify__C": [0.1, 1, 10],
        "classify__max_iter": [1000],
        "classify__class_weight": [None, 'balanced']
    },
    {
        "classify__solver": ['saga'],
        "classify__penalty": ["elasticnet"],
        "classify__l1_ratio": [0, 0.5, 1],
        "classify__C": [0.1, 1, 10],
        "classify__max_iter": [1000],
        "classify__class_weight": [None, 'balanced']
    }
]

grid_2 = GridSearchCV(
    estimator=pipe_2,
    param_grid=lr_param_grid,
    cv=5,
    scoring='f1_macro'
)

grid_2.fit(X_train_val, y_train_val)
print(grid_2.best_params_)

c:\Users\TEXNH MAKPH\Projects\Yandex DS\ds-yandex\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\TEXNH MAKPH\Projects\Yandex DS\ds-yandex\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\TEXNH MAKPH\Projects\Yandex DS\ds-yandex\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penal

{'classify__C': 1, 'classify__class_weight': 'balanced', 'classify__max_iter': 1000, 'classify__penalty': 'l1', 'classify__solver': 'saga'}


c:\Users\TEXNH MAKPH\Projects\Yandex DS\ds-yandex\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\TEXNH MAKPH\Projects\Yandex DS\ds-yandex\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [53]:
best_pipe_2 = clone(pipe_2)
best_pipe_2.set_params(**grid_2.best_params_)
best_pipe_2.fit(X_train_val, y_train_val)

print(classification_report(y_train_val, best_pipe_2.predict(X_train_val)))
print(classification_report(y_test, best_pipe_2.predict(X_test)))

              precision    recall  f1-score   support

           0       0.14      0.37      0.20       160
           1       0.59      0.34      0.43       769
           2       0.48      0.54      0.50       579

    accuracy                           0.42      1508
   macro avg       0.40      0.41      0.38      1508
weighted avg       0.50      0.42      0.43      1508

              precision    recall  f1-score   support

           0       0.10      0.25      0.14        40
           1       0.58      0.34      0.43       193
           2       0.46      0.52      0.49       145

    accuracy                           0.40       378
   macro avg       0.38      0.37      0.35       378
weighted avg       0.48      0.40      0.42       378



c:\Users\TEXNH MAKPH\Projects\Yandex DS\ds-yandex\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\TEXNH MAKPH\Projects\Yandex DS\ds-yandex\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


---